In [1]:
!uv add toyaikit

Resolved 147 packages in 991ms                                       
Prepared 5 packages in 561ms                                                 anthropic            ------------------------------ 1.83 MiB/1.87 MiB           anthropic            ------------------------------ 301.62 KiB/1.87 MiB         anthropic            ------------------------------ 93.62 KiB/1.87 MiB          
Uninstalled 1 package in 1ms
Installed 5 packages in 14ms                                
 ~ ai-engineering-buildcamp-code==0.1.0 (from file:///Users/binayokbhowmik/Training/AI-Courses/Maven/ai-engineering-buildcamp-code)
 + anthropic==1.13.0
 + docstring-parser==0.18.0
 + genai-prices==0.1.10
 + toyaikit==0.0.11


In [2]:
from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import AppendableIndex


reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()

parsed_docs = [doc.parse() for doc in files]
chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)

index = AppendableIndex(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(chunked_docs)

In [3]:
def search(query):
    results = index.search(
        query=query,
        num_results=5
    )
    return results

search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the documentation database for relevant results based on a query string.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The search query to look up in the index"
            }
        },
        "required": [
            "query"
        ]
    }
}

In [4]:
def add_entry(filename, title, description, content):
    entry = {
        'start': 0,
        'content': content,
        'title': title,
        'description': description,
        'filename': filename,
    }
    index.append(entry)
    return "OK"

add_entry_tool = {
    "type": "function",
    "name": "add_entry",
    "description": "Add a new documentation entry to the index.",
    "parameters": {
        "type": "object",
        "properties": {
            "filename": {
                "type": "string",
                "description": "The source filename associated with the entry"
            },
            "title": {
                "type": "string",
                "description": "The title of the documentation entry"
            },
            "description": {
                "type": "string",
                "description": "A short description summarizing the entry"
            },
            "content": {
                "type": "string",
                "description": "The full content of the documentation entry"
            }
        },
        "required": [
            "filename",
            "title",
            "description",
            "content"
        ]
    }
}

In [5]:
instructions = """
You're a documentation assistant. 

Answer the user question using the documentation knowledge base

Make 3 iterations:

1) in the first iteration, perform one search
2) in the second interation, analyze the results from the previous search
   and perform 2 more searches
3) synthesise the results into the output

IMPORTANT: at each step, give an explanation of why you want to perform 
search for this particular search query. It should be 2-3 sentences explaining
the logic of your decision.

Use only facts from the knowledge base when answering.
If you cannot find the answer, inform the user.

Our knowledge base is entirely about Evidently, so you don't need to 
include the word 'evidently' in search results
"""

In [6]:
from toyaikit.tools import Tools

In [7]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)
agent_tools.add_tool(add_entry, add_entry_tool)

In [8]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the documentation database for relevant results based on a query string.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'The search query to look up in the index'}},
   'required': ['query']}},
 {'type': 'function',
  'name': 'add_entry',
  'description': 'Add a new documentation entry to the index.',
  'parameters': {'type': 'object',
   'properties': {'filename': {'type': 'string',
     'description': 'The source filename associated with the entry'},
    'title': {'type': 'string',
     'description': 'The title of the documentation entry'},
    'description': {'type': 'string',
     'description': 'A short description summarizing the entry'},
    'content': {'type': 'string',
     'description': 'The full content of the documentation entry'}},
   'required': ['filename', 'title', 'description', 'content']}}]

In [9]:
from toyaikit.chat.runners import OpenAIResponsesRunner
from toyaikit.llm import OpenAIClient
from openai import OpenAI

llm_client = OpenAIClient(
    model="gpt-4o-mini",
    client=OpenAI()
)

In [10]:
agent = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    # chat_interface=,
    llm_client=llm_client
)

In [11]:
result = agent.loop('how can I create evidently dahsbord?')

In [14]:

print(result.last_message)

Based on the results of the second search, here’s a synthesis of how to create a dashboard.

### Steps for Creating a Dashboard

1. **Overview and Requirements**:
   - A dashboard provides a visual summary of your AI application's performance, allowing you to monitor evaluation results and production quality over time. Each project has its own dashboard, which starts empty. To use it, you need to run evaluations and save at least one report to your project.

2. **Adding Panels**:
   - You can create a dashboard by adding various types of panels to visualize data. Panels can be added either through the user interface or the Python API. Each panel displays specific values retrieved from reports saved in your project. 
   - Depending on your needs, you can create different types of panels (e.g., counters, text panels, pie charts, and line plots).

3. **Organizing with Tabs**:
   - Dashboards can include multiple tabs, allowing you to logically organize your panels. You can add new tabs ei

In [15]:
result.cost.total_cost

Decimal('0.0022455')

In [16]:
from toyaikit.chat.interface import IPythonChatInterface
from toyaikit.chat.runners import DisplayingRunnerCallback

In [17]:
chat_interface = IPythonChatInterface()
runner_callback = DisplayingRunnerCallback(chat_interface=chat_interface) 

In [18]:
result = agent.loop(
    'how can I create evidently dahsbord?',
    callback=runner_callback
)

-> Response received


-> Response received


-> Response received


In [19]:
result2 = agent.loop(
    'show me the code',
    callback=runner_callback,
    previous_messages=result.all_messages
)

-> Response received


In [20]:
agent = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=llm_client
)

In [21]:
result = agent.run()

-> Response received


-> Response received


-> Response received


-> Response received


Chat ended.


In [22]:
result.cost.total_cost

Decimal('0.00511305')